In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm

import sxobsplan 
import ccdproc
from astropy.io import fits
from astropy.wcs import WCS
from astroquery.jplhorizons import Horizons
from astropy.time import Time, TimeDelta
from astropy.coordinates import SkyCoord
from astropy.nddata import Cutout2D
from astropy.constants import c
import astropy.units as u
import astrometry
from astropy.stats import sigma_clip
from photutils.aperture import CircularAperture, CircularAnnulus, aperture_photometry
import sep

import matplotlib.pyplot as plt
import _rcparams # type: ignore
from astropy.visualization import ZScaleInterval

In [ ]:
targetname = "240P"
WORKDIR  = Path.cwd() / ".."
DATADIR = WORKDIR / "data" / "".join(targetname.split())
FIGDIR = DATADIR / "fig_afrho"
FIGDIR.mkdir(parents=True, exist_ok=True)

In [ ]:
allfits = ccdproc.ImageFileCollection(DATADIR, glob_include="*.fits").filter()
summary = allfits.summary.to_pandas()
summary.head()

In [ ]:
phot_target = pd.DataFrame()
obstime = Time(summary['obsjd'], format="jd")

phot_target["file"]     = summary['file'].apply(lambda fpath: Path(fpath).name)
phot_target["des"]      = targetname
phot_target["orb_id"]   = 90001204 # 240P/NEAT (90001203, 90001204)
phot_target["jd"]       = obstime.jd
phot_target["isot"]     = obstime.isot
phot_target["exptime"]  = summary['exptime']
phot_target["egain"]    = summary['gain']
phot_target["pixscale"] = summary['pixscale']  # arcsec / pixel
phot_target["fwhm_pix"] = summary['seeing'] # pixel
phot_target["filter"]   = summary['filter']
phot_target["zpmag"]    = summary['magzp'] # zero point magnitude
phot_target["zpmagerr"] = summary['magzpunc'] # zero point magnitude error

In [ ]:
# 240P/NEAT's orbit at previous apparition
mask_orbid = obstime < Time("2022-01-01")
phot_target.loc[mask_orbid, "orb_id"]  = 90001203
phot_target["orb_id"] = phot_target["orb_id"].astype(int)

In [ ]:
for idx, row in tqdm(phot_target.iterrows(), total=len(phot_target), desc="Fetching ephemerides"):
    
    obj = Horizons(id=row.orb_id, location='I41', epochs=row.jd)
    eph = obj.ephemerides()[0]
    
    phot_target.at[idx, "ra"]          = eph['RA']
    phot_target.at[idx, "dec"]         = eph['DEC']
    phot_target.at[idx, "r"]           = eph['r']
    phot_target.at[idx, "delta"]       = eph['delta']
    phot_target.at[idx, "alpha"]       = eph['alpha']
    phot_target.at[idx, "elong"]       = eph['elong']
    phot_target.at[idx, "sunTargetPA"] = eph['sunTargetPA']
    phot_target.at[idx, "velocityPA"]  = eph['velocityPA']

In [ ]:
phot_target["rho_km"] = 15000 # projected radius km

pixel_scale_km = (phot_target["pixscale"] * ((1*u.arcsec).to(u.rad).value) * phot_target["delta"] * (1*u.au).to(u.km).value) # km/pixel
phot_target["rho_pix"] = phot_target["rho_km"] / pixel_scale_km

# size of background annulus
phot_target["sky_in_pix"]  = 3*phot_target["rho_pix"]
phot_target["sky_out_pix"] = 4*phot_target["rho_pix"] + 20
phot_target["rho_fwhm"] = phot_target["rho_pix"] / phot_target["fwhm_pix"]

In [ ]:
for idx, row in tqdm(phot_target.iterrows(), total=len(phot_target), desc="Performing aperture photometry"):
    
    filepath = DATADIR / row.file
    with fits.open(filepath) as hdul:
        data = hdul[0].data.astype(np.float32)
        hdr = hdul[0].header
        wcs  = WCS(hdul[0].header)
        err  = np.sqrt(data/row.egain + (hdr['READNOI']/row.egain)**2)

    skycoord = SkyCoord(ra=row.ra*u.deg, dec=row.dec*u.deg, frame='icrs')
    xycoord = wcs.world_to_pixel(skycoord)

    # Refine object position with SEP.winpos
    xycoord_winpos = sep.winpos(data, xinit=xycoord[0], yinit=xycoord[1], sig=3*row.fwhm_pix)

    # Define apertures
    aperture = CircularAperture((xycoord_winpos[0], xycoord_winpos[1]), r=row.rho_pix)
    annulus = CircularAnnulus((xycoord_winpos[0], xycoord_winpos[1]), r_in=row.sky_in_pix, r_out=row.sky_out_pix)

    # Perform aperture photometry
    phot_table = aperture_photometry(data, aperture, error=err)

    # Estimate sky background from annulus
    mask = annulus.to_mask(method='center')
    ann_data = mask.multiply(data)
    if ann_data is None:
        msky = np.nan
        ssky = np.nan
        nsky = 0
    else:
        sky_data = ann_data[mask.data==1]
        sky_data_clipped = sigma_clip(sky_data, sigma=3, maxiters=5)
        med = np.ma.median(sky_data_clipped)
        stddev = np.ma.std(sky_data_clipped)
        msky = med
        ssky = stddev
        nsky = sky_data_clipped.count() # number of unmasked pixels

    phot_target.at[idx, "x_center"]       = xycoord_winpos[0]
    phot_target.at[idx, "y_center"]       = xycoord_winpos[1]
    phot_target.at[idx, "aparea"]         = aperture.area
    phot_target.at[idx, "source_sum"]     = phot_table['aperture_sum'][0] - msky * aperture.area
    phot_target.at[idx, "source_sum_err"] = np.sqrt(phot_table['aperture_sum_err'][0]**2 + aperture.area * ssky**2)
    phot_target.at[idx, "snr"]            = phot_target.at[idx, "source_sum"] / phot_target.at[idx, "source_sum_err"]
    phot_target.at[idx, "inst_mag"]       = -2.5 * np.log10(phot_target.at[idx, "source_sum"])
    phot_target.at[idx, "mag_err"]        = 2.5/np.log(10)*(1/phot_target.at[idx, "snr"])

In [ ]:
phot_target["filter_mag"] = phot_target["inst_mag"] + row.zpmag

In [ ]:
# Apparent solar AB magnitude at PS1 filter system (Willmer et al. 2018)
dict_solmag = {
    "ZTF_g": -26.54,
    "ZTF_r": -26.93,
    "ZTF_i": -27.05
}

phot_target["solmag"] = phot_target["filter"].map(dict_solmag)
phot_target["magdiff_sol"] = phot_target["filter_mag"] - phot_target["solmag"]
phot_target["flux_ratio_sol"] = 10**(-0.4*(phot_target["magdiff_sol"])) # in unit of solar flux

In [ ]:
phot_target

In [ ]:
# Afrho calculation
flux_ratio_sol = phot_target["flux_ratio_sol"].to_numpy()
r = phot_target["r"].to_numpy()
delta = phot_target["delta"].to_numpy() * u.au
rho_km = phot_target["rho_km"].to_numpy() * u.km

afrho = ( 4 * (delta**2) * (r**2) / rho_km * flux_ratio_sol ).to_value(u.cm)
phot_target["afrho_cm"] = afrho

In [ ]:
# Correct with phase function

def phase_func_linear(alpha, beta=0.03):
    """Linear phase function correction.
    
    Parameters
    ----------
    alpha : float or ndarray
        Phase angle in degrees.
    beta : float, optional
        Linear phase coefficient in mag/deg. Default is 0.03 mag/deg.
        
    Returns
    -------
    phi : float or ndarray
        Phase function correction factor.
    """
    
    mag_correction = beta * alpha
    
    phi = 10**(0.4 * mag_correction)
    
    return phi

phot_target["phase_corr"] = phase_func_linear(phot_target["alpha"], beta=0.03)
phot_target["afrho_cm_corr"] = phot_target["afrho_cm"] * phot_target["phase_corr"]

In [ ]:
phot_target.to_csv(DATADIR / f"photometry_{targetname}.csv", index=False)

In [ ]:
tp_jd_early = 2458256.7471537665 # time of perihelion (rough estimate)
tp_jd_recent = 2461029.353067453108 

In [ ]:
filter_name = "ZTF_r"

phot_target_early = phot_target[phot_target["orb_id"] == 90001203]
phot_target_early_r = phot_target_early[phot_target_early["filter"] == "ZTF_r"]

phot_target_recent = phot_target[phot_target["orb_id"] == 90001204]
phot_target_recent_r = phot_target_recent[phot_target_recent["filter"] == "ZTF_r"]

In [ ]:
fig = plt.figure(figsize=(10,8))
ax = fig.add_subplot()

ax.scatter(phot_target_early_r["jd"]-tp_jd_early, phot_target_early_r["afrho_cm_corr"], c='k', s=30)
# ax.scatter(phot_target_recent_r["jd"]-tp_jd_recent, phot_target_recent_r["afrho_cm_corr"], c='r', s=30)

ax.set_xlim(0, 400)
ax.set_ylim(0, 500)

ax.set_xlabel(r"$T-T_\mathrm{p}$ (d)")
ax.set_ylabel(r"$A(0\degree) f \rho$ (cm)")

ax.annotate(rf"""{targetname} ({filter_name})
$\rho={phot_target.rho_km.iloc[0]}$ km
$T_p$ = {Time(tp_jd_early, format='jd').isot}""",
            xy=(0.02, 0.95), xycoords='axes fraction', fontsize=16,
            va="top", ha="left",
            family='monospace')

plt.savefig(FIGDIR / f"afrho_{targetname}_{filter_name}_early.png")
plt.show()

In [ ]:
phot_target_recent_r['r']

In [ ]:
fig = plt.figure(figsize=(10,6))
ax = fig.add_subplot()

ax.scatter(phot_target_recent_r["jd"]-tp_jd_recent, phot_target_recent_r["afrho_cm_corr"], c='k', s=30)
# ax.scatter(phot_target_recent_r["jd"]-tp_jd_recent, phot_target_recent_r["afrho_cm_corr"], c='r', s=30)

ax.set_xlim(-150, -30)
ax.set_ylim(0, 600)

ax.set_xlabel(r"$T-T_\mathrm{p}$ (d)")
ax.set_ylabel(r"$A(0\degree) f \rho$ (cm)")

ax.annotate(rf"""{targetname} ({filter_name}), $\rho={phot_target.rho_km.iloc[0]}$ km
$T_p$ = {Time(tp_jd_recent, format='jd').isot}""",
            xy=(0.98, 0.95), xycoords='axes fraction', fontsize=16,
            va="top", ha="right",
            family='monospace')

ax.axvspan(-145, -125, color='gray', alpha=0.3)

plt.savefig(FIGDIR / f"afrho_{targetname}_{filter_name}_recent.png")
plt.show()

In [ ]:
# dict_filter = {
#     'ZTF_g':  {'lambda_eff_AA': 4874, 'lambda_width_AA': 1196},
#     'ZTF_r':  {'lambda_eff_AA': 6417, 'lambda_width_AA': 1418},
#     'ZTF_i':  {'lambda_eff_AA': 7956, 'lambda_width_AA': 1422},
# }

# phot_target["lambda_eff_AA"]   = phot_target["filter"].map(lambda f: dict_filter[f]['lambda_eff_AA'])
# phot_target["lambda_width_AA"] = phot_target["filter"].map(lambda f: dict_filter[f]['lambda_width_AA'])

# f_nu = 3631 * 10**(-0.4 * phot_target["filter_mag"].to_numpy()) * u.Jy # in Jy
# f_lambda =  f_nu * c / (phot_target["lambda_eff_AA"].to_numpy() * u.AA)**2
# flux = f_lambda * (phot_target["lambda_width_AA"].to_numpy()) * u.AA

In [ ]:
###
### All figure to save
###

for idx, row in summary.iterrows():
    
    fpath_fits = row["file"]
    hdul = fits.open(fpath_fits)
    header = hdul[0].header
    wcs = WCS(hdul[0].header)
    obstime = Time(hdul[0].header['OBSJD'], format='jd')
    obj = Horizons(id=90001204, location='I41', epochs=obstime.jd)
    eph = obj.ephemerides().to_pandas().iloc[0]

    fig = plt.figure()
    ax = fig.add_subplot(projection=wcs)
    vmin, vmax = ZScaleInterval().get_limits(hdul[0].data)
    ax.imshow(hdul[0].data, cmap='gray', vmin=vmin, vmax=vmax, origin='lower')

    text_annotate = f"""OBSERVAT = {hdul[0].header['ORIGIN']}
DATE-OBS = {obstime.isot}
FILTER   = {hdul[0].header['FILTER']}
EXPTIME  = {hdul[0].header['EXPTIME']} sec

OBJECT   = {targetname}
Rh       = {eph['r']:.3f} AU
Delta    = {eph['delta']:.3f} AU
Phase    = {eph['alpha']:.1f} deg
Elong    = {eph['elong']:.1f} deg
S-T PA   = {eph.sunTargetPA:.1f} deg
Vel PA   = {eph.velocityPA:.1f} deg
"""

    ax.annotate(text_annotate, xy=(1.05, 0.95), xycoords='axes fraction',
                fontsize=15, color='k', 
                va="top", ha="left", family="monospace")
    
    arrow_style = dict(facecolor="white", edgecolor="white", width=3, head_width=10, head_length=5)
    arrow_text_style = dict(color="white", ha="center", va="center", fontsize=15, weight="bold")
    ny, nx = hdul[0].data.shape
    base_x, base_y = 0.2 * nx, 0.8 * ny
    arrow_length = 0.13 * min(nx, ny)
    # North Arrow
    ax.arrow(base_x, base_y,
             0, -arrow_length, 
             **arrow_style)
    ax.text(base_x, base_y - arrow_length * 1.4,
            'N', **arrow_text_style)  
    
    # East Arrow
    ax.arrow(base_x, base_y,
             -arrow_length, 0,
             **arrow_style)
    ax.text(base_x - arrow_length * 1.4,
            base_y,
            'E', **arrow_text_style)
    # velocity PA Arrow
    vel_pa_rad = np.deg2rad(eph.velocityPA)
    ax.arrow(base_x, base_y,
                -arrow_length * np.sin(vel_pa_rad),
                -arrow_length * np.cos(vel_pa_rad),
                **arrow_style)
    ax.text(base_x - arrow_length * 1.4 * np.sin(vel_pa_rad),
            base_y - arrow_length * 1.4 * np.cos(vel_pa_rad),
            '$-V$', **arrow_text_style)
    # sun Target PA Arrow
    sun_pa_rad = np.deg2rad(eph.sunTargetPA)
    ax.arrow(base_x, base_y,
                -arrow_length * np.sin(sun_pa_rad),
                -arrow_length * np.cos(sun_pa_rad),
                **arrow_style)
    ax.text(base_x - arrow_length * 1.4 * np.sin(sun_pa_rad),
            base_y - arrow_length * 1.4 * np.cos(sun_pa_rad),
            '$-\u2609$', **arrow_text_style)    
    
    plt.savefig(FIGDIR/f"{header['FILENAME']}.png")
    plt.close()  